# Lab 1 - Introduction to Natural Language Processing
## Exploring the SMS Spam Collection

**Objective:** select an online text dataset, load it, and inspect its contents using pandas.

This notebook completes the practical tasks from the supplied Lab 1 handout using SMS messages in place of IMDB reviews. It follows the same exploration steps: `head()`, `describe()`, `info()`, and class counts.

## 1. Background

Natural language processing (NLP) helps computers process and analyse human language. Common applications include translation, information extraction, sentiment analysis, and spam detection.

Human language is challenging because of ambiguity, spelling variations, and context. SMS messages are useful examples because they often contain abbreviations and informal spelling. A **corpus** is a collection of texts; this dataset is a corpus of labelled messages.

A typical NLP workflow includes acquiring text, preprocessing it, converting it into numerical features, and building a model. This introductory lab focuses on acquiring and inspecting the data.

## 2. Task 1 - Find and select a dataset

**Selected dataset:** SMS Spam Collection, from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).

**File type:** a UTF-8 text file with tab-separated fields and no header. Text files are one of the formats listed in the lab task.

**Why this dataset:** it is small enough to inspect easily and shows a practical NLP application: distinguishing normal SMS messages from spam. It contains message text and a category label.

| Column | Meaning |
| --- | --- |
| `label` | `ham` for a normal message or `spam` for an unsolicited message |
| `message` | The original SMS text |

Dataset citation: Almeida, T. & Hidalgo, J. (2011). *SMS Spam Collection* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84

UCI lists a [CC BY 4.0 license](https://creativecommons.org/licenses/by/4.0/). The original source documentation is included in the `data` folder.

## 3. Task 2 - Download and open the dataset

The [official ZIP](https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip) was downloaded and its `SMSSpamCollection` file included as `data/SMSSpamCollection.txt`. The contents are unchanged. This local copy makes the notebook reproducible without downloading the dataset on every run.

Install the repository requirements before running this notebook. NLTK is imported to match the library introduction in the handout; no NLTK corpus download is needed.

In [1]:
from pathlib import Path
import csv
import pandas as pd
import nltk

print("pandas:", pd.__version__)
print("NLTK:", nltk.__version__)

pandas: 3.0.6
NLTK: 3.10.3


In [2]:
# Support launching from either the Lab 1 folder or the repository root.
candidates = [Path("data/SMSSpamCollection.txt"),
              Path("Lab 1/data/SMSSpamCollection.txt")]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from Lab 1 or the repository root.")

# Treat quotation marks as message text, not CSV quoting syntax.
df = pd.read_csv(data_path, sep="\t", header=None,
                 names=["label", "message"], encoding="utf-8",
                 quoting=csv.QUOTE_NONE, keep_default_na=False)
print(f"Loaded {len(df):,} messages and {len(df.columns)} columns.")

Loaded 5,574 messages and 2 columns.


### Preview the first rows

`head()` displays the first five records so we can check that the labels and messages loaded into the correct columns.

In [3]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Descriptive statistics

For text columns, `describe()` reports the number of non-missing entries (`count`), distinct values (`unique`), the most frequent value (`top`), and its frequency (`freq`).

In [4]:
df.describe()

,label,message
count,5574,5574
unique,2,5171
top,ham,"Sorry, I'll call later"
freq,4827,30


### Column information

`info()` shows the row count, column names, non-null counts, data types, and memory usage.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5574 non-null   str  
 1   message  5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


### Class distribution

The handout counts sentiment labels. Here we count `ham` and `spam` because this dataset is for spam classification.

In [6]:
counts = df["label"].value_counts()
print(counts)
distribution = counts.rename("count").to_frame()
distribution["percentage"] = (counts / len(df) * 100).round(2)
distribution

label
ham     4827
spam     747
Name: count, dtype: int64


,count,percentage
label,,
ham,4827,86.6
spam,747,13.4


### Additional data-quality checks

Check both missing values and empty strings, and count repeated label-message pairs. Duplicates are reported and retained; cleaning or training a classifier is outside this lab's required tasks.

In [7]:
quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "blank_strings": df.apply(lambda column: column.str.strip().eq("").sum())
})
print("Duplicate rows:", df.duplicated().sum())
quality

Duplicate rows: 403


,missing_values,blank_strings
label,0,0
message,0,0


## 4. Findings

- The downloaded file contains **5,574 messages** with two columns.
- **4,827 messages (86.6%) are ham** and **747 (13.4%) are spam**, so the classes are imbalanced.
- There are **no missing or blank values** and **403 duplicate rows**. These rows are retained to preserve the source data.
- Message text would be the input for a future classifier, and the `label` column would be its target.

The required dataset selection and loading tasks are complete, along with the exploration demonstrated in the handout. No classifier is trained in this introductory lab.